# HalfCourt - XGBoost Training

Train the XGBoost classifier on the HalfCourt halftime dataset and save `xgboost.joblib`.

## 1. Install dependencies

In [ ]:
!pip install -q xgboost shap scikit-learn pandas pyarrow joblib

## 2. Upload dataset

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

## 3. Set configuration

In [ ]:
PARQUET_PATH = 'halfcourt_training.parquet'
MODEL_OUT = 'xgboost.joblib'
TEST_SIZE = 0.2
RANDOM_STATE = 42

N_ESTIMATORS = 300
MAX_DEPTH = 5
LEARNING_RATE = 0.05
SUBSAMPLE = 0.8
COLSAMPLE_BYTREE = 0.8
EARLY_STOPPING_ROUNDS = 20

## 4. Load dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet(PARQUET_PATH)
df['game_date'] = pd.to_datetime(df['game_date'])
df = df.sort_values('game_date').reset_index(drop=True)

print(f'Dataset shape : {df.shape}')
print(f'Date range    : {df["game_date"].min().date()} to {df["game_date"].max().date()}')
print(f'Home win rate : {df["home_win"].mean():.3f}')

## 5. Temporal train / test split

In [ ]:
FEATURE_COLS = [
    'ht_score_diff', 'q1_diff', 'q2_diff', 'momentum_shift',
    'home_fg_pct', 'home_3p_pct', 'home_ft_pct', 'home_efg_pct', 'home_ts_pct',
    'away_fg_pct', 'away_3p_pct', 'away_ft_pct', 'away_efg_pct', 'away_ts_pct',
    'home_to', 'away_to', 'home_ast_to_ratio', 'away_ast_to_ratio',
    'home_oreb_pct', 'away_oreb_pct', 'reb_diff',
    'home_pace_est', 'away_pace_est',
    'home_bench_pts', 'away_bench_pts', 'home_bench_ratio', 'away_bench_ratio',
    'home_starters_3foul', 'away_starters_3foul', 'foul_diff',
    'home_star1_pts', 'home_star1_ast', 'home_star_foul_flag',
    'away_star1_pts', 'away_star1_ast', 'away_star_foul_flag',
    'is_home', 'home_rest_days', 'away_rest_days',
    'home_back2back', 'away_back2back', 'is_playoffs',
]
LABEL_COL = 'home_win'

split_idx = int(len(df) * (1 - TEST_SIZE))
train_df  = df.iloc[:split_idx]
test_df   = df.iloc[split_idx:]

X_train = train_df[FEATURE_COLS].values.astype(np.float32)
y_train = train_df[LABEL_COL].values.astype(np.float32)
X_test  = test_df[FEATURE_COLS].values.astype(np.float32)
y_test  = test_df[LABEL_COL].values.astype(np.float32)

print(f'Train samples : {len(X_train)}')
print(f'Test samples  : {len(X_test)}')

## 6. Train XGBoost

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    subsample=SUBSAMPLE,
    colsample_bytree=COLSAMPLE_BYTREE,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    device='cuda',
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)
print('Training complete.')

## 7. Evaluation report

In [ ]:
import shap
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, brier_score_loss

proba = model.predict_proba(X_test)[:, 1]
preds = (proba >= 0.5).astype(int)

print('=== HalfCourt Model Evaluation ===')
print(f'Model type     : xgboost')
print(f'Train samples  : {len(X_train)}')
print(f'Test samples   : {len(X_test)}')
print()
print(f'Accuracy       : {accuracy_score(y_test, preds):.3f}')
print(f'AUC-ROC        : {roc_auc_score(y_test, proba):.3f}')
print(f'F1 Score       : {f1_score(y_test, preds):.3f}')
print(f'Brier Score    : {brier_score_loss(y_test, proba):.3f}')
print()

explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
mean_abs    = np.abs(shap_values).mean(axis=0)
ranked      = sorted(zip(FEATURE_COLS, mean_abs), key=lambda x: x[1], reverse=True)[:10]

print('Top 10 Features by SHAP importance:')
for rank, (feat, val) in enumerate(ranked, 1):
    print(f'  {rank:2d}. {feat:<30} {val:.3f}')

## 8. SHAP summary plot

In [ ]:
import matplotlib.pyplot as plt

shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLS, show=False)
plt.tight_layout()
plt.savefig('xgboost_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to xgboost_shap_summary.png')

## 9. Save model

In [ ]:
import joblib

joblib.dump(model, MODEL_OUT)
print(f'Model saved -> {MODEL_OUT}')

## 10. Download

Download `xgboost.joblib` and place it in `models/saved/` in your local repo.

In [ ]:
files.download(MODEL_OUT)
files.download('xgboost_shap_summary.png')